# 7. Credit valuation adjustment (CVA)

**Level:** L3. **Tags:** `xva`, `cva`, `counterparty-credit`.
**Library:** `qp_core.xva` (`python/core`).

**Plain idea.** If the other side of a trade defaults while it owes you
money, you lose that money less what you recover. CVA is the expected loss:
add up, over future dates, what you would be owed times the chance that the
default happens then.

**Formal statement** (Gregory, *The xVA Challenge*, 4th ed., 2020, the CVA
chapter): with discounted expected exposure $EE^*(t)$, survival $S(t)$ and
recovery $R$,
$\mathrm{CVA} = (1-R)\int_0^T EE^*(t)\,(-dS(t)) \approx
(1-R)\sum_k EE^*(t_k)\,[S(t_{k-1}) - S(t_k)]$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math

import numpy as np
from qp_core.xva.adjustments import HazardCurve, cva  # noqa: F401

## Task 1: CVA of a flat exposure

A counterparty with a CDS spread of 120bp and recovery 40% has, by the
credit triangle, hazard $\lambda = s/(1-R) = 2\%$. Take a flat discounted
exposure of 1,000,000 on a quarterly grid to 5 years. Store the CVA as `c`.

In [ ]:
times = np.linspace(0.0, 5.0, 21)
ee = np.full_like(times, 1_000_000.0)
curve = ...
c = ...
# Exercise: replace the ... above, then run the check cell below.

**Independent check.** With a flat exposure the sum telescopes:
$\mathrm{CVA} = (1-R)\,E\,(1 - e^{-\lambda T})$, exactly, on any grid.

In [ ]:
lam = 0.012 / 0.60
assert abs(c - 0.60 * 1_000_000.0 * (1 - math.exp(-lam * 5.0))) < 1e-6

## Task 2: an amortising exposure

Exposure that runs down linearly from 1,000,000 to 0 (a loan repaid evenly)
must cost less than the flat one. Store it as `c_amort`.

In [ ]:
c_amort = ...
# Exercise: replace the ... above, then run the check cell below.

**Independent check.** The integral of $(1-R)E(1-t/T)\lambda e^{-\lambda t}$
in closed form, which the quarterly sum approaches from below (the sum
takes the exposure at the end of each quarter).

In [ ]:
exact = 0.60 * 1_000_000.0 * (1 - (1 - math.exp(-lam * 5.0)) / (lam * 5.0))
assert 0 < c_amort < exact < c
assert abs(c_amort - exact) / exact < 0.06

**Takeaways.** CVA is exposure times default probability, priced. The
library computes the exposure profile of a swap under Hull-White
(`qp_core.xva.exposure.swap_exposure`) and adds DVA and FVA.